# Train B1e (NTIP)

Trains **B1e**, the platform's primary ML model: a small multilingual encoder
fine-tuned on the repository's **synthetic** training data (DDL-020, ML layer
design section 5.3). Nothing here uses real call data.

**Before running:** *Runtime → Change runtime type → T4 GPU*. Then
*Runtime → Run all*. Training takes a few minutes on a T4.

The notebook:
1. clones the repository at `REF` and installs the backend;
2. trains B1e on the generated dataset's **dev** split only (the regression
   split is kept unseen for calibration);
3. evaluates it against B0 (rules) and B1 on the curated **dev and regression**
   splits, which it never saw (the held-out test split is not touched);
4. packages the model directory and the evaluation reports for download.

The model directory's `manifest.json` records the commit, data digest, base
model revision, hyperparameters, seed, hardware and the sha256 of every file.
The platform refuses a model whose files do not match it.

In [ ]:
# The repository commit or branch to train from. Use a commit id for a run you
# intend to publish, so the manifest points at exactly the code that trained it.
REF = "main"
REPO = "https://github.com/izuchukwumba/ng911_triage_intelligence_platform.git"

In [ ]:
import subprocess, sys
print(sys.version)
assert sys.version_info >= (3, 12), "The backend needs Python 3.12 or newer"
try:
    print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "no GPU")
except FileNotFoundError:
    print("No GPU: training will run on CPU and take much longer. "
          "Runtime -> Change runtime type -> T4 GPU.")

In [ ]:
%cd /content
!rm -rf ntip && git clone --quiet {REPO} ntip
%cd /content/ntip
!git checkout --quiet {REF} && git log --oneline -1
# Colab already has PyTorch; pip keeps it when it satisfies the pins.
!pip install --quiet -e "backend[ml,ml-encoder]"

In [ ]:
%cd /content/ntip/backend
!rm -rf /content/b1e
# Train on the dev split only: the regression split stays unseen so the scores can be
# calibrated on it afterwards (ML layer design section 6, milestone ML-9).
!python -m ntip.ml.train_b1e --dataset ../data/eval/generated-2.2.0-seed42.jsonl --split dev --out /content/b1e

In [ ]:
import json
manifest = json.load(open("/content/b1e/manifest.json"))
print(json.dumps({k: manifest[k] for k in
      ("model_version", "base_model", "training_data", "environment", "git_commit", "weights_digest")},
      indent=2))

## Evaluate on data B1e never saw

Curated dev and regression splits (30 author-labelled cases, outside the
generator's families). This compares B0, B1, B1e and the hybrids. Results on
30 synthetic cases are a development view, not a claim of accuracy.

In [ ]:
%cd /content/ntip/backend
!rm -rf /content/results && mkdir -p /content/results
!python -m ntip.evaluation.compare --dataset ../data/eval/curated.jsonl \
    --split dev --split regression --with-b1e /content/b1e --out /content/results

## Download

`b1e-model.zip` is the model directory. Keep it somewhere you control (for
example attached to a GitHub release); do not commit it to git. To use it
locally, unzip it and point the platform at it:

```
NTIP_MODEL=b1e-encoder NTIP_B1E_PATH=path/to/b1e uvicorn ntip.ml.service:create_inference_app --factory --port 8100
python -m ntip.evaluation.compare --dataset ../data/eval/curated.jsonl --split dev --split regression --with-b1e path/to/b1e
```

`b1e-results.zip` holds the evaluation reports (JSON and Markdown) to commit
under `docs/evaluation/results/`.

In [ ]:
%cd /content
!rm -f b1e-model.zip b1e-results.zip
!cd /content && zip -qr b1e-model.zip b1e && zip -qr b1e-results.zip results
!ls -lh /content/*.zip
from google.colab import files
files.download("/content/b1e-model.zip")
files.download("/content/b1e-results.zip")